## Transport data

#### imports

In [3]:
import pandas as pd
import numpy as np
import scipy as sp
import itertools
import datetime
import plotly.express as px
import plotly.graph_objects as go

#### variables

In [4]:
hours_in_a_day = 24

In [5]:
regions = {
    'South America': ['Colombia','Ecuador','Peru','Bolivia','Brasil'],
    'Central America': ['Guatemala','Honduras','Nicaragua','El Salvador'],
    'Southern Africa': ['South Africa','Malawi','Mozambique','Eswatini','Lesotho']
}

In [6]:
mini_bus = ['chapa','kombi','mini bus']

In [7]:
countries = {
    'Colombia' : {
        'color': '#EFCA08'
    },
    'Ecuador' : {
        'color': '#FF9000'
    },
    'Peru' : {
        'color': '#DB5461'
    },
    'Bolivia' : {
        'color': '#A4243B'
    },
    'Brasil' : {
        'color': '#94BFA7'
    },
    'Guatemala' : {
        'color': '#86A5D9'
    },
    'Honduras' : {
        'color': '#0B3954'
    },
    'Nicaragua' : {
        'color': '#57C4E5'
    },
    'El Salvador': {
        'color': '#273043'
    },
    'South Africa': {
        'color': '#D4E09B'
    },
    'Malawi': {
        'color': '#D72638'
    },
    'Mozambique': {
        'color': '#2D5C1E'
    },
    'Eswatini': {
        'color': '#AFA3B8'
    },
    'Lesotho': {
        'color': '#8EF9F3'
    }
}

#### functions

In [8]:
# formatting columns to be nicer to work with
def format_names(df):
    for col in df.columns:
        df = df.rename(columns={col: col.replace(' ','_')})

    return df

In [11]:
transport = pd.read_csv('../data/transport_data.csv')
transport = format_names(transport)
transport['date'] = pd.to_datetime(transport['date'], format='%m/%d/%Y')

# have to get rid of the random extra column i added for some reason
transport = transport.drop('Unnamed:_8', axis=1)

transport.head()

,city,destination,region,country,date,approximate_time,type,time_of_day
0,Cartagena,Santa Marta,Bolivar,Colombia,2024-03-06,4.00,collectivo,day
1,Santa Marta,Sierra Nevada Park,Magdalena,Colombia,2024-03-08,1.00,van,day
2,Sierra Nevada Park,Tayrona Park,Magdalena,Colombia,2024-03-13,0.50,van,day
3,Tayrona Park,Santa Marta,Magdalena,Colombia,2024-03-14,0.75,bus,day
4,Santa Marta,Taganga,Magdalena,Colombia,2024-03-14,0.25,collectivo,day


In [10]:
# fixing hitch hiking split
transport['type'] = transport['type'].str.replace('hitch hiking','hitchhiking')
transport['type'] = transport['type'].str.replace('bus ','bus')

In [9]:
transport.type.unique()

array(['collectivo', 'van', 'bus', 'flight', 'teleferico', 'jeep', 'boat',
       'car share', 'train', 'chicken bus', 'shuttle', 'hitchhiking',
       'truck', 'car', 'mini bus', 'chapa', 'kombi'], dtype=object)

In [10]:
transport[transport.type == 'truck']

,city,destination,region,country,date,approximate_time,type,time_of_day,Unnamed:_8
194,La Unión,Volcán Conchagua,La Unión,El Salvador,2025-04-01,1.0,truck,day,NaN
195,Volcán Conchagua,La Unión,La Unión,El Salvador,2025-04-02,1.0,truck,day,NaN


### Time to look at transportation

In [11]:
# remove malawi boat
mb_idx = transport[(transport.country == 'Malawi') & (transport.type == 'boat')]
transport = transport.drop(mb_idx.index[0], axis=0)

In [12]:
ttt = transport.approximate_time.sum()
print ('I spent {} hours in total in different types of transport - {} days'.format(ttt, np.round(ttt/hours_in_a_day, 1)))


I spent 1048.75 hours in total in different types of transport - 43.7 days


In [13]:
transport_types = pd.DataFrame(columns=['type','number','time_sum','time_max','time_avg','time_min'])
for x in transport.type.unique():
    clip = transport[transport.type == x]
    transport_types.loc[len(transport_types)] = [x, 
                                                clip.shape[0], 
                                                clip.approximate_time.sum(), 
                                                clip.approximate_time.max(), 
                                                clip.approximate_time.mean(),
                                                clip.approximate_time.min()
                                               ]

In [14]:
transport_types

,type,number,time_sum,time_max,time_avg,time_min
0,collectivo,28,52.75,7.0,1.883929,0.25
1,van,19,68.00,10.0,3.578947,0.50
2,bus,102,498.25,22.0,4.884804,0.50
3,flight,19,84.25,15.0,4.434211,0.75
4,teleferico,2,2.00,1.0,1.000000,1.00
5,jeep,3,9.00,8.0,3.000000,0.50
6,boat,24,30.50,4.0,1.270833,0.50
7,car share,4,14.50,5.0,3.625000,1.50
8,train,1,13.00,13.0,13.000000,13.00
9,chicken bus,15,20.50,3.5,1.366667,0.25


In [15]:
# for viz purposes will probably put these all together
mini_bus_time = transport[transport.type.isin(mini_bus)].approximate_time.sum()
print ('Africa has many different names for their local minibuses, total time spent: {} hours'.format(mini_bus_time))

Africa has many different names for their local minibuses, total time spent: 41.75 hours


#### Night bus suffering!!

In [16]:
night_bus = transport[(transport.type.isin(['bus','van','shuttle'])) & (transport.time_of_day == 'night')]

print ('{} buses'.format(night_bus.shape[0]))
print ('{} hours (~{} days)'.format(night_bus.approximate_time.sum(), np.round((night_bus.approximate_time.sum()/hours_in_a_day),1)))

nb_regions = pd.DataFrame(columns=['region','count','time_sum'])
for x in regions.keys():
    clip = night_bus[night_bus.country.isin(regions[x])]
    nb_regions.loc[len(nb_regions)] = [x, clip.shape[0], clip.approximate_time.sum()]

nb_regions

26 buses
267.5 hours (~11.1 days)


,region,count,time_sum
0,South America,22,223.5
1,Central America,3,32.0
2,Southern Africa,1,12.0


#### transport over time
- highlight chart with different colors for different countries

In [17]:
def add_rect(country, x_range, y_max):
    fig.add_shape(
        type='rect',
        x0=x_range[0], y0=0, x1=x_range[1], y1=y_max,
        fillcolor=countries[country]['color'],
        opacity=0.4,
        layer='below',
        line_width=0,
    )

In [18]:
transport.head(10)

,city,destination,region,country,date,approximate_time,type,time_of_day,Unnamed:_8
0,Cartagena,Santa Marta,Bolivar,Colombia,2024-03-06,4.00,collectivo,day,NaN
1,Santa Marta,Sierra Nevada Park,Magdalena,Colombia,2024-03-08,1.00,van,day,country
2,Sierra Nevada Park,Tayrona Park,Magdalena,Colombia,2024-03-13,0.50,van,day,Colombia
3,Tayrona Park,Santa Marta,Magdalena,Colombia,2024-03-14,0.75,bus,day,Ecuador
4,Santa Marta,Taganga,Magdalena,Colombia,2024-03-14,0.25,collectivo,day,Peru
5,Taganga,Santa Marta,Magdalena,Colombia,2024-03-16,0.25,collectivo,day,Bolivia
6,Santa Marta,Medellín,Antioquia,Colombia,2024-03-16,1.50,flight,night,Brasil
7,Medellín,Guatapé,Antioquia,Colombia,2024-03-19,2.00,bus,day,Panama
8,Guatapé,Medellín,Antioquia,Colombia,2024-03-19,2.00,bus,day,Canada
9,Medellín,Parque Arvi,Antioquia,Colombia,2024-03-22,1.00,teleferico,day,Guatemala


In [19]:
# need to group by because sometimes take different transport on one day
t_grp = transport.groupby(['country','date','approximate_time']).sum()

# creating the base line chart
fig = px.line(
    transport, 
    x='date', 
    y='approximate_time', 
    labels={
         'date': 'Date',
         'approximate_time': 'Hours'
    },
    title='Hours spent in transportation over time'
)

fig.update_traces(line_color='black')

y_max = transport.approximate_time.max()

# adding the different colours to represent different countries
for c in list(itertools.chain.from_iterable(regions.values())):
    clip = transport[transport.country == c]
    # having an issue with the boxes not matching up because there is a day between travels. need to get date from idx before
    # deal with this later

    # Ecuador case! two different parts of the trip
    if c == 'Ecuador':
        # first leg
        x_range = [clip[clip.region != 'Galapagos'].date.min(), clip[clip.region != 'Galapagos'].date.max()]
        add_rect(c, x_range, y_max)

        # second leg
        x_range = [clip[clip.region == 'Galapagos'].date.min(), clip[clip.region == 'Galapagos'].date.max()]
        add_rect(c, x_range, y_max)

    else: 
        x_range = [clip.date.min(), clip.date.max()]
        add_rect(c, x_range, y_max)

# fig.update_xaxes(tickformat="%B %Y")
fig.show()



In [20]:
#maybe make a chart showing time for the individual countries?

#### Days spent in each country
- need to find the rows where we switch from one country to another
- will probably need to embed a rule that notes me pausing travel and then coming back, going back and forth between countries

In [21]:
transport

,city,destination,region,country,date,approximate_time,type,time_of_day,Unnamed:_8
0,Cartagena,Santa Marta,Bolivar,Colombia,2024-03-06,4.00,collectivo,day,NaN
1,Santa Marta,Sierra Nevada Park,Magdalena,Colombia,2024-03-08,1.00,van,day,country
2,Sierra Nevada Park,Tayrona Park,Magdalena,Colombia,2024-03-13,0.50,van,day,Colombia
3,Tayrona Park,Santa Marta,Magdalena,Colombia,2024-03-14,0.75,bus,day,Ecuador
4,Santa Marta,Taganga,Magdalena,Colombia,2024-03-14,0.25,collectivo,day,Peru
...,...,...,...,...,...,...,...,...,...
303,Cape Town,Hermanus,Western Cape,South Africa,2025-09-24,1.50,car,day,NaN
304,Hermanus,Cape Town,Western Cape,South Africa,2025-09-25,2.00,car,day,NaN
305,Cape Town,Addis Ababa,Oromia,Ethiopia,2025-09-25,6.50,flight,day,NaN
306,Addis Ababa,Rome,Lazio,Italy,2025-09-25,6.50,flight,day,NaN


In [22]:
transit_countries = ['Canada', 'Panama', 'United States', 'Ethiopia', 'Italy']

In [23]:
timeline = pd.DataFrame(columns=['country', 'entry_date', 'exit_date', 'num_of_days'])
for idx, row in transport.iterrows():
    # handle the entrance into the first country          
    if idx == 0:
        timeline.loc[0] = [row.country, row.date, None, None]
                        
    else:        
        # crossed a border
        if prev_country != row.country:
            date_diff = row.date - prev_date
           
            # make sure that it is not a break from travel by making sure the exit and entrance dates are close
            # have to make this value larger because I need to make sure that when i left brasil gets caught as the last 5 days I did no transporting
            if date_diff.days < 6:
                # TO DO: do i need to consider type of travel? or is that too into the specifics

                # update the missing values for the country you just left
                num_of_days = row.date - timeline.loc[timeline.shape[0]-1,'entry_date']
                timeline.loc[timeline.shape[0]-1, ['exit_date', 'num_of_days']] = [row.date.date(), num_of_days.days]

                # update the base values for the country you just entered
                timeline.loc[timeline.shape[0]] = [row.country, row.date, None, None]     
                
    # update values
    prev_country = row.country
    prev_date = row.date

In [24]:
# trim the transit countries out of the data set
timeline = timeline[~timeline.country.isin(transit_countries)]

In [30]:
transport[transport.country == 'South Africa']

,city,destination,region,country,date,approximate_time,type,time_of_day,Unnamed:_8
213,Newark,Cape Town,Western Cape,South Africa,2025-04-16,15.00,flight,night,NaN
214,Cape Town,Hermanus,Western Cape,South Africa,2025-04-17,2.50,car,day,NaN
215,Hermanus,Karoo,Western Cape,South Africa,2025-04-18,4.50,car,day,NaN
216,Karoo,Hermanus,Western Cape,South Africa,2025-04-20,4.50,car,day,NaN
217,Hermanus,Cape Town,Western Cape,South Africa,2025-04-22,1.50,car,day,NaN
218,Cape Town,Hermanus,Western Cape,South Africa,2025-04-22,2.50,car,day,NaN
219,Hermanus,Bettysbaai,Western Cape,South Africa,2025-04-25,0.75,car,day,NaN
220,Bettysbaai,Hermanus,Western Cape,South Africa,2025-04-25,0.75,car,day,NaN
221,Hermanus,Gansbaai,Western Cape,South Africa,2025-04-27,0.75,car,day,NaN
222,Gansbaai,Hermanus,Western Cape,South Africa,2025-04-27,0.75,car,day,NaN


In [25]:
timeline

,country,entry_date,exit_date,num_of_days
0,Colombia,2024-03-06,2024-04-18,43
1,Ecuador,2024-04-18,2024-05-13,25
2,Peru,2024-05-13,2024-07-02,50
3,Bolivia,2024-07-02,2024-08-15,44
4,Brasil,2024-08-15,2024-12-19,126
7,Ecuador,2025-01-17,2025-02-15,29
8,Guatemala,2025-02-15,2025-03-08,21
9,Honduras,2025-03-08,2025-03-17,9
10,Nicaragua,2025-03-17,2025-04-01,15
11,El Salvador,2025-04-01,2025-04-16,15


In [26]:
# total amount of days stayed in each country
timeline.groupby('country').num_of_days.sum().reset_index()

,country,num_of_days
0,Bolivia,44
1,Brasil,126
2,Colombia,43
3,Ecuador,54
4,El Salvador,15
5,Eswatini,9
6,Guatemala,21
7,Honduras,9
8,Lesotho,4
9,Malawi,16


In [27]:
# save the timeline to a csv
timeline.to_csv('../data/country_timeline.csv', index=False)

In [28]:
timeline.num_of_days.sum()

539